# GALVAT Trial 7 Roboflow Source Checkpoint Builder

This notebook restores the current Trial 7 dataset checkpoint from Google Drive, lets you add new Roboflow source URLs, validates image and YOLO label quality, appends only accepted examples into the fixed 44-class Trial 7 ontology, audits what is still missing, and writes a new checkpoint zip to Drive.

Primary model metric later remains **GALVAT weighted score**. Do not train until the final audit says the dataset has acceptable class coverage.

## 1. Restore The Existing Checkpoint

This restores the zip you saved before sleeping. If the exact filename changes, edit `CHECKPOINT_STEM` below.

In [ ]:
from pathlib import Path
import shutil, zipfile, json, yaml, re, hashlib, math, random, datetime
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from google.colab import drive

random.seed(42)

drive.mount('/content/drive')

CHECKPOINT_STEM = 'galvat_trial7_dataset_working_checkpoint_2026-06-11_05-37-59'
DRIVE_CHECKPOINT_DIR = Path('/content/drive/MyDrive/galvat_trial7_dataset_checkpoints')
RESTORE_PARENT = Path('/content/galvat_trial7_dataset_working_restore')

candidates = [
    DRIVE_CHECKPOINT_DIR / f'{CHECKPOINT_STEM}.zip',
    DRIVE_CHECKPOINT_DIR / CHECKPOINT_STEM,
]
if not any(p.exists() for p in candidates):
    candidates.extend(sorted(DRIVE_CHECKPOINT_DIR.glob(f'{CHECKPOINT_STEM}*.zip')))

checkpoint_zip = next((p for p in candidates if p.exists()), None)
assert checkpoint_zip is not None, f'Could not find checkpoint in {DRIVE_CHECKPOINT_DIR}'

if RESTORE_PARENT.exists():
    shutil.rmtree(RESTORE_PARENT)
RESTORE_PARENT.mkdir(parents=True, exist_ok=True)

print('Restoring checkpoint:', checkpoint_zip)
with zipfile.ZipFile(checkpoint_zip, 'r') as z:
    z.extractall(RESTORE_PARENT)

data_yaml_candidates = list(RESTORE_PARENT.rglob('dataset/data.yaml'))
assert data_yaml_candidates, 'No dataset/data.yaml found after unzip.'

WORK_DS = data_yaml_candidates[0].parent
WORK_ROOT = WORK_DS.parent
WORK_META = WORK_ROOT / 'metadata'
WORK_META.mkdir(parents=True, exist_ok=True)

with open(WORK_DS / 'data.yaml', 'r') as f:
    data_yaml = yaml.safe_load(f)

names_obj = data_yaml['names']
if isinstance(names_obj, dict):
    FINAL_NAMES = [names_obj[i] if i in names_obj else names_obj[str(i)] for i in range(len(names_obj))]
else:
    FINAL_NAMES = list(names_obj)

final_id = {name: i for i, name in enumerate(FINAL_NAMES)}

print('WORK_ROOT:', WORK_ROOT)
print('WORK_DS:', WORK_DS)
print('WORK_META:', WORK_META)
print('Classes:', len(FINAL_NAMES))
print(FINAL_NAMES)

## 2. Current Needs

The last known checkpoint still needed these classes most:

**Missing entirely:** `tupperware`, `lemon`, `drinking_glass`, `paper_towel_dispenser`, `urinal`, `hearing_aid`, `deodorant`, `toothpaste`, `door_frame`, `thermostat`, `exit_sign`, `bathroom_sign`, `step_curb`, `razor`.

**Low count:** `measuring_cup`, `light_switch`, `railing`, `salt_pepper_shaker`, `wallet`, `soap_dispenser`, `pan`, `electrical_outlet`, `jug`.

Put new Roboflow URLs in the next cell. Keep source names short and unique.

In [ ]:
# 3. ADD NEW ROBOFLOW SOURCES HERE
#
# To add a source:
# 1. Paste the Roboflow download URL into url.
# 2. Set enabled=True.
# 3. Confirm target_class is one of FINAL_NAMES.
# 4. Add source label aliases that might appear in data.yaml.

ROBOFLOW_SOURCES = [
    {
        'enabled': False,
        'source_name': 'paper_towel_dispenser',
        'target_class': 'paper_towel_dispenser',
        'url': '',
        'aliases': ['paper towel dispenser', 'paper_towel_dispenser', 'paper-towel-dispenser'],
        'target_total_images': 1000,
    },
    {
        'enabled': False,
        'source_name': 'hearing_aid',
        'target_class': 'hearing_aid',
        'url': '',
        'aliases': ['hearing aid', 'hearing_aid', 'hearing-aid'],
        'target_total_images': 1000,
    },
    {
        'enabled': False,
        'source_name': 'bathroom_sign',
        'target_class': 'bathroom_sign',
        'url': '',
        'aliases': ['bathroom sign', 'bathroom_sign', 'restroom sign', 'toilet sign'],
        'target_total_images': 1000,
    },
    {
        'enabled': False,
        'source_name': 'step_curb',
        'target_class': 'step_curb',
        'url': '',
        'aliases': ['step', 'curb', 'kerb', 'step_curb', 'stairs'],
        'target_total_images': 1200,
    },
    {
        'enabled': False,
        'source_name': 'exit_sign',
        'target_class': 'exit_sign',
        'url': '',
        'aliases': ['exit sign', 'exit_sign'],
        'target_total_images': 800,
    },
    {
        'enabled': False,
        'source_name': 'urinal',
        'target_class': 'urinal',
        'url': '',
        'aliases': ['urinal'],
        'target_total_images': 700,
    },
    {
        'enabled': False,
        'source_name': 'deodorant',
        'target_class': 'deodorant',
        'url': '',
        'aliases': ['deodorant'],
        'target_total_images': 800,
    },
    {
        'enabled': False,
        'source_name': 'toothpaste',
        'target_class': 'toothpaste',
        'url': '',
        'aliases': ['toothpaste'],
        'target_total_images': 800,
    },
    {
        'enabled': False,
        'source_name': 'razor',
        'target_class': 'razor',
        'url': '',
        'aliases': ['razor', 'razors', 'shaver'],
        'target_total_images': 800,
    },
    {
        'enabled': False,
        'source_name': 'drinking_glass',
        'target_class': 'drinking_glass',
        'url': '',
        'aliases': ['drinking glass', 'drinking_glass', 'glass'],
        'target_total_images': 1000,
    },
    {
        'enabled': False,
        'source_name': 'tupperware',
        'target_class': 'tupperware',
        'url': '',
        'aliases': ['tupperware', 'food container', 'storage container', 'plastic container'],
        'target_total_images': 800,
    },
    {
        'enabled': False,
        'source_name': 'lemon',
        'target_class': 'lemon',
        'url': '',
        'aliases': ['lemon'],
        'target_total_images': 800,
    },
    {
        'enabled': False,
        'source_name': 'thermostat',
        'target_class': 'thermostat',
        'url': '',
        'aliases': ['thermostat'],
        'target_total_images': 700,
    },
    {
        'enabled': False,
        'source_name': 'door_frame',
        'target_class': 'door_frame',
        'url': '',
        'aliases': ['door frame', 'door_frame', 'door'],
        'target_total_images': 800,
    },
]

enabled_sources = [s for s in ROBOFLOW_SOURCES if s.get('enabled') and s.get('url')]
print('Enabled Roboflow sources:', len(enabled_sources))
for s in enabled_sources:
    assert s['target_class'] in final_id, f"Unknown target_class: {s['target_class']}"
    print('-', s['source_name'], '->', s['target_class'])

## 4. Download And Inspect Roboflow Sources

Run this after adding sources above. The output table must look sane before merging.

In [ ]:
import requests

ROBOSRC_ROOT = Path('/content/roboflow_sources_trial7_continue')
ROBOSRC_ROOT.mkdir(parents=True, exist_ok=True)
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def norm_name(x):
    return re.sub(r'[^a-z0-9]+', ' ', str(x).lower()).strip()

def download_roboflow_source(src, overwrite=False):
    out_dir = ROBOSRC_ROOT / src['source_name']
    zip_path = ROBOSRC_ROOT / f"{src['source_name']}.zip"
    if overwrite and out_dir.exists():
        shutil.rmtree(out_dir)
    if out_dir.exists() and list(out_dir.rglob('data.yaml')):
        print('Already downloaded:', src['source_name'])
        return out_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    print('Downloading', src['source_name'])
    with requests.get(src['url'], stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(zip_path, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True, desc=src['source_name']) as bar:
            for chunk in r.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(out_dir)
    zip_path.unlink(missing_ok=True)
    return out_dir

def find_data_yaml(source_dir):
    candidates = list(Path(source_dir).rglob('data.yaml'))
    if not candidates:
        raise FileNotFoundError(f'No data.yaml found in {source_dir}')
    return sorted(candidates, key=lambda p: len(p.parts))[0]

def load_yolo_names(data_yaml_path):
    with open(data_yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    names = data.get('names')
    if isinstance(names, dict):
        names = [names[i] if i in names else names[str(i)] for i in range(len(names))]
    return data, list(names)

for src in enabled_sources:
    download_roboflow_source(src, overwrite=False)

inspect_rows = []
for src in enabled_sources:
    source_dir = ROBOSRC_ROOT / src['source_name']
    data_yaml_path = find_data_yaml(source_dir)
    data, names = load_yolo_names(data_yaml_path)
    alias_norms = {norm_name(a) for a in src['aliases']}
    mapped_ids = [i for i, n in enumerate(names) if norm_name(n) in alias_norms]
    inferred_single_class = len(names) == 1 and not mapped_ids

    split_counts = {}
    label_counts = Counter()
    for split in ['train', 'valid', 'test']:
        img_dir = data_yaml_path.parent / split / 'images'
        lbl_dir = data_yaml_path.parent / split / 'labels'
        imgs = list(img_dir.glob('*')) if img_dir.exists() else []
        split_counts[split] = len([p for p in imgs if p.suffix.lower() in IMAGE_EXTS])
        if lbl_dir.exists():
            for lbl in lbl_dir.glob('*.txt'):
                for line in lbl.read_text().splitlines():
                    if line.strip():
                        cls_id = int(float(line.split()[0]))
                        label_counts[names[cls_id] if 0 <= cls_id < len(names) else f'BAD_ID_{cls_id}'] += 1

    inspect_rows.append({
        'source_name': src['source_name'],
        'target_class': src['target_class'],
        'data_yaml': str(data_yaml_path),
        'source_classes': names,
        'mapped_source_class_ids': mapped_ids,
        'single_class_fallback_would_be_used': inferred_single_class,
        'train_images': split_counts.get('train', 0),
        'valid_images': split_counts.get('valid', 0),
        'test_images': split_counts.get('test', 0),
        'source_label_counts': dict(label_counts),
    })

inspect_df = pd.DataFrame(inspect_rows)
display(inspect_df)
inspect_df.to_csv(WORK_META / 'roboflow_continue_source_inspection.csv', index=False)
print('Saved:', WORK_META / 'roboflow_continue_source_inspection.csv')

## 5. Curate And Append Passing Images

This cell validates YOLO labels, rejects bad coordinates, rejects poor-quality images, rejects images with too many labels, maps labels into the fixed Trial 7 class IDs, and appends selected images into the working dataset.

In [ ]:
SMALL_CLASSES = {
    'keys', 'medicine_bottle', 'electrical_outlet', 'light_switch',
    'toothpaste', 'deodorant', 'razor', 'hearing_aid', 'wallet',
    'thermostat', 'salt_pepper_shaker', 'measuring_cup',
}
THIN_CLASSES = {'cane', 'railing', 'keys', 'writing_device', 'step_curb'}

def current_class_image_counts():
    counts = Counter()
    for split in ['train', 'valid', 'test']:
        for lbl_path in (WORK_DS / split / 'labels').glob('*.txt'):
            seen = set()
            for line in lbl_path.read_text().splitlines():
                if line.strip():
                    cls_id = int(float(line.split()[0]))
                    if 0 <= cls_id < len(FINAL_NAMES):
                        seen.add(FINAL_NAMES[cls_id])
            for cls in seen:
                counts[cls] += 1
    return counts

def split_for_roboflow(source_name, file_name):
    h = int(hashlib.md5(f'roboflow-continue-{source_name}-{file_name}'.encode()).hexdigest(), 16) % 100
    if h < 80:
        return 'train'
    if h < 90:
        return 'valid'
    return 'test'

def image_quality_ok(img_path):
    try:
        im = Image.open(img_path).convert('L')
        arr = np.asarray(im.resize((256, 256)))
        mean = float(arr.mean())
        std = float(arr.std())
        sharpness = float(np.abs(np.diff(arr.astype(np.float32), axis=1)).mean() + np.abs(np.diff(arr.astype(np.float32), axis=0)).mean())
        reasons = []
        if mean < 30 or mean > 225:
            reasons.append('bad_brightness')
        if std < 18:
            reasons.append('low_contrast')
        if sharpness < 5.5:
            reasons.append('likely_blurry')
        if img_path.stat().st_size < 8000:
            reasons.append('tiny_file')
        return len(reasons) == 0, {'brightness': mean, 'contrast': std, 'sharpness': sharpness, 'reject_reason': ','.join(reasons)}
    except Exception as e:
        return False, {'brightness': None, 'contrast': None, 'sharpness': None, 'reject_reason': f'decode_error:{e}'}

def parse_source_labels(lbl_path, names, mapped_ids, target_class, allow_single_class_fallback):
    lines_out = []
    reject_reasons = []
    raw_lines = lbl_path.read_text().splitlines() if lbl_path.exists() else []
    if not raw_lines:
        return [], ['empty_label']
    for line in raw_lines:
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5:
            reject_reasons.append('bad_column_count')
            continue
        try:
            src_cls = int(float(parts[0]))
            vals = [float(x) for x in parts[1:]]
            x, y, w, h = vals
        except Exception:
            reject_reasons.append('parse_error')
            continue
        if src_cls < 0 or src_cls >= len(names):
            reject_reasons.append('bad_class_id')
            continue
        is_target = src_cls in mapped_ids
        if allow_single_class_fallback and len(names) == 1:
            is_target = True
        if not is_target:
            reject_reasons.append(f'unknown_source_class:{names[src_cls]}')
            continue
        if not all(math.isfinite(v) for v in vals):
            reject_reasons.append('nonfinite_coord')
            continue
        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
            reject_reasons.append('coord_out_of_range')
            continue
        area = w * h
        aspect = max(w / max(h, 1e-6), h / max(w, 1e-6))
        min_area = 0.00015 if target_class in SMALL_CLASSES else 0.0006
        max_aspect = 25 if target_class in THIN_CLASSES else 15
        if area < min_area:
            reject_reasons.append('box_too_small')
            continue
        if area > 0.85:
            reject_reasons.append('box_too_large')
            continue
        if aspect > max_aspect:
            reject_reasons.append('box_aspect_extreme')
            continue
        lines_out.append(f'{final_id[target_class]} {x:.6f} {y:.6f} {w:.6f} {h:.6f}')
    if not lines_out and not reject_reasons:
        reject_reasons.append('no_usable_target_label')
    return lines_out, reject_reasons

current_counts = current_class_image_counts()
selected_rows = []
rejected_rows = []
summary_rows = []

for src in enabled_sources:
    target_class = src['target_class']
    source_name = src['source_name']
    needed = max(0, int(src['target_total_images']) - current_counts.get(target_class, 0))
    if needed <= 0:
        print(f'Skipping {source_name}: target already met for {target_class}')
        continue
    data_yaml_path = find_data_yaml(ROBOSRC_ROOT / source_name)
    data, names = load_yolo_names(data_yaml_path)
    alias_norms = {norm_name(a) for a in src['aliases']}
    mapped_ids = [i for i, n in enumerate(names) if norm_name(n) in alias_norms]
    allow_single_class_fallback = len(names) == 1 and not mapped_ids
    candidates = []
    for original_split in ['train', 'valid', 'test']:
        img_dir = data_yaml_path.parent / original_split / 'images'
        lbl_dir = data_yaml_path.parent / original_split / 'labels'
        if not img_dir.exists():
            continue
        for img_path in img_dir.iterdir():
            if img_path.suffix.lower() not in IMAGE_EXTS:
                continue
            lbl_path = lbl_dir / f'{img_path.stem}.txt'
            lines_out, label_reasons = parse_source_labels(lbl_path, names, mapped_ids, target_class, allow_single_class_fallback)
            if not lines_out:
                rejected_rows.append({'source_name': source_name, 'target_class': target_class, 'file_name': img_path.name, 'reason': ';'.join(sorted(set(label_reasons)))})
                continue
            pixel_ok, q = image_quality_ok(img_path)
            if not pixel_ok:
                rejected_rows.append({'source_name': source_name, 'target_class': target_class, 'file_name': img_path.name, 'reason': q['reject_reason']})
                continue
            label_count = len(lines_out)
            if label_count > 12:
                rejected_rows.append({'source_name': source_name, 'target_class': target_class, 'file_name': img_path.name, 'reason': f'too_many_labels:{label_count}'})
                continue
            score = min(q['contrast'], 80) * 0.2 + min(q['sharpness'], 40) * 0.8 + (8 if 1 <= label_count <= 4 else -4) - 2 * max(0, label_count - 4)
            candidates.append({'source_name': source_name, 'target_class': target_class, 'original_split': original_split, 'img_path': img_path, 'label_lines': lines_out, 'label_count': label_count, 'score': score, **q, 'allow_single_class_fallback': allow_single_class_fallback, 'source_classes': names})
    candidates = sorted(candidates, key=lambda r: r['score'], reverse=True)
    picked = candidates[:needed]
    for row in tqdm(picked, desc=f'Appending {source_name}->{target_class}'):
        img_path = row['img_path']
        out_split = split_for_roboflow(source_name, img_path.name)
        stem_hash = hashlib.md5(str(img_path).encode()).hexdigest()[:10]
        out_stem = f'rf2_{source_name}_{stem_hash}_{img_path.stem}'
        out_img = WORK_DS / out_split / 'images' / f'{out_stem}{img_path.suffix.lower()}'
        out_lbl = WORK_DS / out_split / 'labels' / f'{out_stem}.txt'
        shutil.copy2(img_path, out_img)
        out_lbl.write_text('\n'.join(row['label_lines']) + '\n')
        selected_rows.append({'source': 'roboflow_continue', 'source_name': source_name, 'target_class': target_class, 'split': out_split, 'original_split': row['original_split'], 'file_name': out_img.name, 'label_count': row['label_count'], 'score': row['score'], 'brightness': row['brightness'], 'contrast': row['contrast'], 'sharpness': row['sharpness'], 'single_class_fallback_used': row['allow_single_class_fallback'], 'source_classes': row['source_classes']})
    current_counts[target_class] += len(picked)
    summary_rows.append({'source_name': source_name, 'target_class': target_class, 'needed_before_import': needed, 'accepted_candidates': len(candidates), 'selected': len(picked), 'target_total_images': src['target_total_images'], 'total_after_import_estimate': current_counts[target_class], 'single_class_fallback_used': allow_single_class_fallback, 'source_classes': names})

selected_df = pd.DataFrame(selected_rows)
rejected_df = pd.DataFrame(rejected_rows)
source_summary_df = pd.DataFrame(summary_rows)
selected_df.to_csv(WORK_META / 'roboflow_continue_selected_import_manifest.csv', index=False)
rejected_df.to_csv(WORK_META / 'roboflow_continue_rejected_import_manifest.csv', index=False)
source_summary_df.to_csv(WORK_META / 'roboflow_continue_import_summary.csv', index=False)

print('Selected images:', len(selected_df))
display(source_summary_df)
print('Top rejection reasons:')
display(rejected_df['reason'].value_counts().head(30) if len(rejected_df) else pd.Series(dtype=int))

## 6. Audit What Is Still Missing

Run this after every source import. Send the missing/low-count output back for the next decision.

In [ ]:
def audit_working_dataset():
    with open(WORK_DS / 'data.yaml', 'r') as f:
        data_yaml = yaml.safe_load(f)
    names_obj = data_yaml['names']
    names = [names_obj[i] if i in names_obj else names_obj[str(i)] for i in range(len(names_obj))]
    split_rows = []
    box_counts = Counter()
    image_counts = Counter()
    empty_counts = Counter()
    bad_rows = []
    for split in ['train', 'valid', 'test']:
        img_files = [p for p in (WORK_DS / split / 'images').glob('*') if p.suffix.lower() in IMAGE_EXTS]
        for img_path in img_files:
            lbl_path = WORK_DS / split / 'labels' / f'{img_path.stem}.txt'
            lines = lbl_path.read_text().splitlines() if lbl_path.exists() else []
            seen = set()
            valid_count = 0
            for i, line in enumerate(lines):
                if not line.strip():
                    continue
                parts = line.split()
                if len(parts) != 5:
                    bad_rows.append((split, img_path.name, i + 1, 'bad_column_count', line))
                    continue
                try:
                    cls_id = int(float(parts[0])); x, y, w, h = [float(v) for v in parts[1:]]
                except Exception:
                    bad_rows.append((split, img_path.name, i + 1, 'parse_error', line))
                    continue
                if cls_id < 0 or cls_id >= len(names):
                    bad_rows.append((split, img_path.name, i + 1, 'bad_class_id', line))
                    continue
                if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                    bad_rows.append((split, img_path.name, i + 1, 'bad_coords', line))
                    continue
                cls = names[cls_id]
                box_counts[(split, cls)] += 1
                seen.add(cls)
                valid_count += 1
            if valid_count == 0:
                empty_counts[split] += 1
            for cls in seen:
                image_counts[(split, cls)] += 1
        split_rows.append({'split': split, 'images': len(img_files), 'empty_label_images': empty_counts[split]})
    class_rows = []
    for cls in names:
        class_rows.append({'class': cls, 'train_boxes': box_counts[('train', cls)], 'valid_boxes': box_counts[('valid', cls)], 'test_boxes': box_counts[('test', cls)], 'total_boxes': sum(box_counts[(s, cls)] for s in ['train', 'valid', 'test']), 'train_images': image_counts[('train', cls)], 'valid_images': image_counts[('valid', cls)], 'test_images': image_counts[('test', cls)], 'total_images': sum(image_counts[(s, cls)] for s in ['train', 'valid', 'test'])})
    split_summary = pd.DataFrame(split_rows)
    class_summary = pd.DataFrame(class_rows).sort_values('total_images')
    bad_labels_df = pd.DataFrame(bad_rows, columns=['split', 'file_name', 'line_num', 'reason', 'line'])
    split_summary.to_csv(WORK_META / 'latest_split_summary.csv', index=False)
    class_summary.to_csv(WORK_META / 'latest_class_summary.csv', index=False)
    bad_labels_df.to_csv(WORK_META / 'latest_bad_labels.csv', index=False)
    return split_summary, class_summary, bad_labels_df

split_summary, class_summary, bad_labels_df = audit_working_dataset()
display(split_summary)
display(class_summary)
print('Bad label rows:', len(bad_labels_df))
if len(bad_labels_df):
    display(bad_labels_df.head(50))

missing = class_summary[class_summary['total_images'] == 0]
low_100 = class_summary[(class_summary['total_images'] > 0) & (class_summary['total_images'] < 100)]
low_300 = class_summary[(class_summary['total_images'] >= 100) & (class_summary['total_images'] < 300)]

print('\nStill missing:')
display(missing[['class', 'total_images']])
print('\nVery low count under 100 images:')
display(low_100[['class', 'total_images', 'total_boxes']])
print('\nLow count 100-299 images:')
display(low_300[['class', 'total_images', 'total_boxes']])

needed_report = {
    'missing': missing['class'].tolist(),
    'very_low_under_100': low_100['class'].tolist(),
    'low_100_to_299': low_300['class'].tolist(),
}
(WORK_META / 'latest_needed_report.json').write_text(json.dumps(needed_report, indent=2))
print('Saved needed report:', WORK_META / 'latest_needed_report.json')

## 7. Save A New Checkpoint Zip To Drive

Run this at the end of the session so `/content` can disappear safely.

In [ ]:
STAMP = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
DRIVE_OUT = Path('/content/drive/MyDrive/galvat_trial7_dataset_checkpoints')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
ZIP_LOCAL = Path(f'/content/galvat_trial7_dataset_working_checkpoint_continue_{STAMP}.zip')
ZIP_DRIVE = DRIVE_OUT / ZIP_LOCAL.name

checkpoint_summary = {
    'created_at': STAMP,
    'source_checkpoint_restored_from': str(checkpoint_zip),
    'work_root': str(WORK_ROOT),
    'work_dataset': str(WORK_DS),
    'work_metadata': str(WORK_META),
    'primary_metric_for_future_training': 'GALVAT weighted score',
    'status': 'dataset_build_checkpoint_not_final_until_missing_classes_are_resolved',
}
(WORK_META / 'checkpoint_continue_summary.json').write_text(json.dumps(checkpoint_summary, indent=2))

if ZIP_LOCAL.exists():
    ZIP_LOCAL.unlink()

INTERNAL_ROOT = Path('galvat_trial7_dataset_working')
with zipfile.ZipFile(ZIP_LOCAL, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    for path in tqdm(list(WORK_ROOT.rglob('*')), desc='Zipping checkpoint'):
        if path.is_file():
            z.write(path, INTERNAL_ROOT / path.relative_to(WORK_ROOT))

print('Local zip:', ZIP_LOCAL)
print('Zip size GB:', ZIP_LOCAL.stat().st_size / 1e9)
print('Copying to Drive:', ZIP_DRIVE)
shutil.copy2(ZIP_LOCAL, ZIP_DRIVE)
print('Saved:', ZIP_DRIVE)